In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib_venn import venn2
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
adata_md = sc.read_h5ad('hm.md_tsv_umap.h5ad')
adata_hc = sc.read_h5ad('hm.hc_tsv_umap.h5ad')
print(f'MD: {adata_md.shape}')
print(f'HC: {adata_hc.shape}')
md_bc = set(adata_md.obs_names)
hc_bc = set(adata_hc.obs_names)
print(f'\nMD unique cells: {len(md_bc)}')
print(f'HC unique cells: {len(hc_bc)}')
print(f'Shared: {len(md_bc & hc_bc)}')
print(f'MD only: {len(md_bc - hc_bc)}')
print(f'HC only: {len(hc_bc - md_bc)}')
# Annotate calling_status (shared vs method-only)
adata_md = adata_md.copy()
adata_md.obs['calling_status'] = 'shared'
adata_md.obs.loc[list(md_bc - hc_bc), 'calling_status'] = 'MD_only'
adata_md.obs['calling_status'] = pd.Categorical(
    adata_md.obs['calling_status'], categories=['shared', 'MD_only'], ordered=True
)
adata_hc = adata_hc.copy()
adata_hc.obs['calling_status'] = 'shared'
adata_hc.obs.loc[list(hc_bc - md_bc), 'calling_status'] = 'HC_only'
adata_hc.obs['calling_status'] = pd.Categorical(
    adata_hc.obs['calling_status'], categories=['shared', 'HC_only'], ordered=True
)
print(f"\n✅ calling_status annotated. MD: {dict(adata_md.obs['calling_status'].value_counts())}")
print(f"   HC: {dict(adata_hc.obs['calling_status'].value_counts())}")

In [ ]:
colors_dict = {'shared': '#B0B0B0', 'MD_only': '#4C78A8'}
adata_md.uns['calling_status_colors'] = [colors_dict[x] for x in adata_md.obs['calling_status'].cat.categories]
fig, ax = plt.subplots(figsize=(5, 4))
sc.pl.umap(
    adata_md,
    color='calling_status',
    legend_loc='right margin',
    size=15,
    frameon=True,
    title='Calling status — MD reference',
    ax=ax,
    show=False,
)
for collection in ax.collections:
    collection.set_rasterized(True)
plt.tight_layout()
plt.show()
#note4fig: fig5d_part2

In [ ]:
colors_dict = {'shared': '#B0B0B0', 'HC_only': '#F58518'}
adata_hc.uns['calling_status_colors'] = [colors_dict[x] for x in adata_hc.obs['calling_status'].cat.categories]
fig, ax = plt.subplots(figsize=(5, 4))
sc.pl.umap(
    adata_hc,
    color='calling_status',
    legend_loc='right margin',
    size=15,
    frameon=True,
    title='Calling status — HC reference',
    ax=ax,
    show=False,
)
for collection in ax.collections:
    collection.set_rasterized(True)
plt.tight_layout()
plt.show()
#note4fig: fig5d_part1

In [ ]:
from sklearn.neighbors import NearestNeighbors
def compute_neighbor_consistency(adata_ref, only_label):
    """Count shared neighbors out of 50 nearest for method-only cells."""
    X_pca = adata_ref.obsm['X_pca'][:, :30]
    nbrs = NearestNeighbors(n_neighbors=51, metric='euclidean')
    nbrs.fit(X_pca)
    distances, indices = nbrs.kneighbors(X_pca)
    only_mask = adata_ref.obs['calling_status'] == only_label
    only_indices = np.where(only_mask)[0]
    counts = []
    for idx in only_indices:
        nbr_idx = indices[idx][1:]  # skip self
        statuses = adata_ref.obs['calling_status'].iloc[nbr_idx]
        counts.append((statuses == 'shared').sum())
    return np.array(counts)

In [ ]:
md_nbrs = compute_neighbor_consistency(adata_md, 'MD_only')
hc_nbrs = compute_neighbor_consistency(adata_hc, 'HC_only')
print(f'MD-only: mean={md_nbrs.mean():.1f}/50 shared neighbors, n={len(md_nbrs)}')
print(f'HC-only: mean={hc_nbrs.mean():.1f}/50 shared neighbors, n={len(hc_nbrs)}')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(md_nbrs, bins=25, color='#4C78A8', alpha=0.7, density=True, edgecolor='white')
axes[0].axvline(np.median(md_nbrs), color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Shared neighbors (out of 50)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'MD-only (median={np.median(md_nbrs):.0f})')
axes[1].hist(hc_nbrs, bins=25, color='#F58518', alpha=0.7, density=True,edgecolor='white')
axes[1].axvline(np.median(hc_nbrs), color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Shared neighbors (out of 50)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'HC-only (median={np.median(hc_nbrs):.0f})')
plt.tight_layout()
plt.show()
#note4fig: fig5e

In [ ]:
color_md = '#4C78A8'
color_knee = '#F58518'
color_shared = '#B0B0B0'
# Load raw called cells from original TSV files
# ModuleDrops: call==cell from run1.barcodes.tsv
df_barcodes = pd.read_csv('run1.barcodes.tsv', sep='\t')
md_bc_raw = set(df_barcodes[df_barcodes['call'] == 'cell']['barcode'])
# Knee: cells.knee.txt (one barcode per line)
hc_bc_raw = set(pd.read_csv('cells.knee.txt', header=None)[0])
print(f'ModuleDrops called: {len(md_bc_raw):,}')
print(f'Knee called:        {len(hc_bc_raw):,}')
print(f'Shared:             {len(md_bc_raw & hc_bc_raw):,}')
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
v = venn2([md_bc_raw, hc_bc_raw], set_labels=('ModuleDrops', 'Knee'), ax=ax)
if v.get_patch_by_id('10') is not None:
    v.get_patch_by_id('10').set_color(color_md)
    v.get_patch_by_id('10').set_alpha(0.55)
if v.get_patch_by_id('01') is not None:
    v.get_patch_by_id('01').set_color(color_knee)
    v.get_patch_by_id('01').set_alpha(0.55)
if v.get_patch_by_id('11') is not None:
    v.get_patch_by_id('11').set_color(color_shared)
    v.get_patch_by_id('11').set_alpha(0.65)
for text in v.set_labels:
    if text is not None:
        text.set_fontsize(10)
for text in v.subset_labels:
    if text is not None:
        text.set_fontsize(10)
ax.set_title('Cell calling overlap', fontsize=11)
plt.tight_layout()
plt.show()
#note4fig: fig5c

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import FuncFormatter, LogLocator, LogFormatterMathtext

counts_file = "counts.tsv"
barcode_file = "run1.barcodes.tsv"
# Number of top-ranked barcodes
N_TOP = 30000
# Number of linear barcode-rank bins
N_RANK_BINS = 50
# LOWESS smoothing parameter
LOWESS_FRAC = 0.03

counts = pd.read_csv(counts_file, sep="\t")

counts = counts.sort_values("umi_count", ascending=False).reset_index(drop=True)

counts["barcode_rank"] = np.arange(len(counts)) + 1

n_top = min(N_TOP, len(counts))
plot_df = counts.iloc[:n_top].copy()

bc = pd.read_csv(barcode_file, sep="\t")

call_info = bc[["barcode", "call"]].drop_duplicates(subset="barcode")

plot_df = plot_df.merge(call_info, on="barcode", how="left")

plot_df["is_cell"] = plot_df["call"] == "cell"

plot_df = plot_df[plot_df["umi_count"] > 0].copy()
# 10. LOWESS
# x = linear barcode rank
# y = actual UMI count
# IMPORTANT:
# We do NOT log-transform UMI before LOWESS.
# The logarithmic appearance comes from the y-axis scale.
lowess_fit = lowess(
    endog=plot_df["umi_count"], exog=plot_df["barcode_rank"], frac=LOWESS_FRAC, it=2, return_sorted=True
)
lowess_x = lowess_fit[:, 0]
lowess_y = lowess_fit[:, 1]
# 11. Linear barcode-rank bins
# 50 bins across 0–50,000
# Each bin contains approximately 1,000 barcodes.
plot_df["rank_bin"] = pd.cut(plot_df["barcode_rank"], bins=N_RANK_BINS, include_lowest=True)
# 12. Calculate cell fraction
# Fraction = number of call == cell
#              all barcodes
# within each barcode-rank bin.
fraction_df = (
    plot_df.groupby("rank_bin", observed=True)
    .agg(barcode_rank=("barcode_rank", "mean"), cell_fraction=("is_cell", "mean"), n_barcodes=("is_cell", "size"))
    .reset_index(drop=True)
)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(lowess_x, lowess_y, linewidth=1.0, color="black", zorder=4, label="LOWESS")
# 16. X axis
# Linear barcode rank
# 0  1  2  3  4  5
# corresponding to
# 0  1  2  3  4  5 × 10^4
ax.set_xlim(0, N_TOP)
ax.set_xticks(np.arange(0, N_TOP + 1, 10000))
def rank_formatter(x, pos):
    return f"{x / 1e4:g}"
ax.xaxis.set_major_formatter(FuncFormatter(rank_formatter))
ax.set_xlabel(r"Barcode rank ($\times 10^4$)")

ax.set_yscale("log")
ax.set_ylim(50, 1e5)
ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,)))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.set_ylabel("UMI counts")

ax2 = ax.twinx()
# 19. Cell fraction bars
# Use exact bin edges so there are NO gaps.
bin_edges = np.linspace(0, N_TOP, N_RANK_BINS + 1)
ax2.bar(
    bin_edges[:-1],
    fraction_df["cell_fraction"],
    width=np.diff(bin_edges),
    align="edge",
    alpha=0.30,
    linewidth=0,
    edgecolor="none",
    zorder=1,
    label="Cell fraction",
)

ax2.set_ylabel("Fraction called as cell")
ax2.set_ylim(0, 1)
ax2.set_yticks(np.arange(0.2, 1.01, 0.2))
ax2.set_yticklabels([f"{x:.1f}" for x in np.arange(0.2, 1.01, 0.2)])

ax.grid(axis="both", alpha=0.15, linewidth=0.6, zorder=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower left", frameon=False, handlelength=2.5)

fig.tight_layout()


plt.show()
plt.close(fig)
#note4fig: fig5a_part2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import FuncFormatter, LogLocator, LogFormatterMathtext

counts_file = "counts.tsv"
# Number of top-ranked barcodes to plot
N_TOP = 30000
# Number of linear barcode-rank bins
N_RANK_BINS = 50
# Number of called barcodes
N_CALLED = 8477
# LOWESS smoothing
LOWESS_FRAC = 0.03
# Box line width
BOX_LINEWIDTH = 0.8

counts = pd.read_csv(counts_file, sep="\t")

counts = counts.sort_values("umi_count", ascending=False).reset_index(drop=True)

counts["barcode_rank"] = np.arange(len(counts)) + 1

n_top = min(N_TOP, len(counts))
plot_df = counts.iloc[:n_top].copy()
# 5. Define calling status
# IMPORTANT:
# No run1.barcodes.tsv is used.
# The first 5770 barcodes are called cells.
plot_df["is_cell"] = plot_df["barcode_rank"] <= N_CALLED

plot_df = plot_df[plot_df["umi_count"] > 0].copy()
# 7. LOWESS
# x = linear barcode rank
# y = actual UMI count
# No log transformation before LOWESS.
lowess_fit = lowess(
    endog=plot_df["umi_count"], exog=plot_df["barcode_rank"], frac=LOWESS_FRAC, it=2, return_sorted=True
)
lowess_x = lowess_fit[:, 0]
lowess_y = lowess_fit[:, 1]

plot_df["rank_bin"] = pd.cut(plot_df["barcode_rank"], bins=N_RANK_BINS, include_lowest=True)
# 9. Calculate cell fraction
# Since the bins are based on linear barcode rank,
# each bin contains approximately the same number
# of barcodes.
fraction_df = (
    plot_df.groupby("rank_bin", observed=True)
    .agg(barcode_rank=("barcode_rank", "mean"), cell_fraction=("is_cell", "mean"), n_barcodes=("is_cell", "size"))
    .reset_index(drop=True)
)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(lowess_x, lowess_y, linewidth=1.0, color="black", zorder=4, label="LOWESS")

ax.set_xlim(0, N_TOP)
ax.set_xticks(np.arange(0, N_TOP + 1, 10000))
def rank_formatter(x, pos):
    return f"{x / 1e4:g}"
ax.xaxis.set_major_formatter(FuncFormatter(rank_formatter))
ax.set_xlabel(r"Barcode rank ($\times 10^4$)")

ax.set_yscale("log")
ax.set_ylim(50, 1e5)
ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,)))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.set_ylabel("UMI counts")

ax2 = ax.twinx()
# 16. Cell fraction bars
# Exact bin edges → NO gaps between bars.
bin_edges = np.linspace(0, N_TOP, N_RANK_BINS + 1)
ax2.bar(
    bin_edges[:-1],
    fraction_df["cell_fraction"],
    width=np.diff(bin_edges),
    align="edge",
    alpha=0.30,
    linewidth=0,
    edgecolor="none",
    zorder=1,
    label="Cell fraction",
)

ax2.set_ylabel("Fraction called as cell")
ax2.set_ylim(0, 1)
ax2.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])

ax.grid(axis="both", alpha=0.15, linewidth=0.6, zorder=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(BOX_LINEWIDTH)
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(BOX_LINEWIDTH)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower left", frameon=False, handlelength=2.5)

fig.tight_layout()


plt.show()
plt.close(fig)
#note4fig: fig5a_part1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import re
from scipy.sparse import csr_matrix
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

In [ ]:
species_df = pd.read_csv('bc_species.tsv', sep='\t')
print(f'Species TSV: {len(species_df)} barcodes')
print(f'Unique species: {species_df["species"].nunique()}')
species_list = pd.read_csv('species_list.tsv', sep='\t', header=None, names=['mgyg', 'species'])
all_species = sorted(species_list['species'].unique().tolist())
print(f'Allowed species: {len(all_species)}')

In [ ]:
from matplotlib import cm, colors as mcolors
palette_colors = []
cmap = cm.get_cmap('tab20', 20)
palette_colors.extend([mcolors.rgb2hex(cmap(i)) for i in range(20)])
cmap = cm.get_cmap('Set2', 12)
palette_colors.extend([mcolors.rgb2hex(cmap(i)) for i in range(12)])
species_colors_unified = {}
for i, sp in enumerate(all_species):
    species_colors_unified[sp] = palette_colors[i]
print(f'Palette built for {len(species_colors_unified)} species')

In [ ]:
import scanpy as sc
adata = sc.read_h5ad("hm.md_tsv_umap.h5ad")

In [ ]:
n_species = adata.obs['species'].nunique()
species_counts = adata.obs['species'].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata.obs['species_plot'] = adata.obs['species'].astype(str)
adata = adata[~adata.obs['species'].isin(rare_species)].copy()
fig, ax = plt.subplots(figsize=(11, 5))
sc.pl.umap(
    adata,
    color='species_plot',
    palette=species_colors_unified,
    legend_loc='right margin',
    size=15,
    frameon=True,
    title='Species UMAP — HM MD (TSV)',
    ax=ax,
    show=False
)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f'Number of species: {n_species}')
plt.tight_layout()
plt.show()
#note4fig: fig5b_part2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import re
from scipy.sparse import csr_matrix
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

In [ ]:
species_df = pd.read_csv('bc_species.tsv', sep='\t')
print(f'Species TSV: {len(species_df)} barcodes')
print(f'Unique species: {species_df["species"].nunique()}')
species_list = pd.read_csv('species_list.tsv', sep='\t', header=None, names=['mgyg', 'species'])
all_species = sorted(species_list['species'].unique().tolist())
print(f'Allowed species: {len(all_species)}')

In [ ]:
from matplotlib import cm, colors as mcolors
palette_colors = []
cmap = cm.get_cmap('tab20', 20)
palette_colors.extend([mcolors.rgb2hex(cmap(i)) for i in range(20)])
cmap = cm.get_cmap('Set2', 12)
palette_colors.extend([mcolors.rgb2hex(cmap(i)) for i in range(12)])
species_colors_unified = {}
for i, sp in enumerate(all_species):
    species_colors_unified[sp] = palette_colors[i]
print(f'Palette built for {len(species_colors_unified)} species')

In [ ]:
import scanpy as sc
adata = sc.read_h5ad("hm.hc_tsv_umap.h5ad")

In [ ]:
n_species = adata.obs['species'].nunique()
species_counts = adata.obs['species'].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata.obs['species_plot'] = adata.obs['species'].astype(str)
adata = adata[~adata.obs['species'].isin(rare_species)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
sc.pl.umap(
    adata,
    color='species_plot',
    palette=species_colors_unified,
    legend_loc='right margin',
    size=15,
    frameon=True,
    title='Species UMAP — HM HC (TSV)',
    ax=ax,
    show=False
)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f'Number of species: {n_species}')
plt.tight_layout()
plt.show()
#note4fig: fig5b_part1

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter
sc.settings.verbosity = 2
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

In [ ]:
def load_tsv(path):
    """Load tri-format tsv (gene, cell, count) into AnnData."""
    df = pd.read_csv(path, sep='\t')
    mat = df.pivot_table(index='cell', columns='gene', values='count', fill_value=0)
    adata = sc.AnnData(mat.astype('float32'))
    adata.layers['counts'] = adata.X.copy()
    adata.obs['total_umis'] = adata.X.sum(axis=1).A1 if hasattr(adata.X, 'A1') else adata.X.sum(axis=1)
    return adata
def filter_coding_genes(adata, gene_biotype):
    """Keep only protein_coding genes."""
    keep = [g for g in adata.var_names if gene_biotype.get(g) == 'protein_coding']
    adata = adata[:, keep].copy()
    return adata
def add_gene_names(adata, gene_name_map):
    adata.var['gene_name'] = adata.var_names.map(gene_name_map)
    # Fill missing names with gene_id
    missing = adata.var['gene_name'].isna()
    adata.var.loc[missing, 'gene_name'] = adata.var_names[missing]
    return adata
def annotate_calling_status(adata_md, adata_hc):
    """Annotate shared / method-only cells for both methods.
    Returns (adata_md_annotated, adata_hc_annotated, shared_set, md_only_set, hc_only_set)
    """
    md_bc = set(adata_md.obs_names)
    hc_bc = set(adata_hc.obs_names)
    shared = md_bc & hc_bc
    md_only = md_bc - hc_bc
    hc_only = hc_bc - md_bc
    # MD reference
    adata_md = adata_md.copy()
    adata_md.obs['calling_status'] = 'shared'
    adata_md.obs.loc[list(md_only), 'calling_status'] = 'MD_only'
    adata_md.obs['calling_status'] = pd.Categorical(
        adata_md.obs['calling_status'], categories=['shared', 'MD_only'], ordered=True
    )
    # HC reference
    adata_hc = adata_hc.copy()
    adata_hc.obs['calling_status'] = 'shared'
    adata_hc.obs.loc[list(hc_only), 'calling_status'] = 'HC_only'
    adata_hc.obs['calling_status'] = pd.Categorical(
        adata_hc.obs['calling_status'], categories=['shared', 'HC_only'], ordered=True
    )
    return adata_md, adata_hc, shared, md_only, hc_only
def plot_status_umap(adata_ref_md, adata_ref_hc, sample_name, axes):
    """Plot calling status UMAP for both references side by side."""
    colors_md = {'shared': 'lightgrey', 'MD_only': '#4C78A8'}
    colors_hc = {'shared': 'lightgrey', 'HC_only': '#F58518'}
    # MD reference
    adata_ref_md.uns['calling_status_colors'] = [
        colors_md[x] for x in adata_ref_md.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(
        adata_ref_md,
        color='calling_status',
        ax=axes[0],
        show=False,
        size=30,
        frameon=True,
        title=f'{sample_name} — MD reference',
    )
    # HC reference
    adata_ref_hc.uns['calling_status_colors'] = [
        colors_hc[x] for x in adata_ref_hc.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(
        adata_ref_hc,
        color='calling_status',
        ax=axes[1],
        show=False,
        size=30,
        frameon=True,
        title=f'{sample_name} — HC reference',
    )
    # Rasterize
    for ax in axes:
        for collection in ax.collections:
            collection.set_rasterized(True)
# Build gene_biotype and gene_name_map from Pcopri/Pcopri.gff3
import re as _re
gene_biotype = {}
gene_name_map = {}
_gff = '/Users/Apple/Downloads/ko_plot/fig5/Pcopri/Pcopri.gff3'
with open(_gff) as _f:
    for _line in _f:
        if _line.startswith('#'): continue
        _cols = _line.rstrip().split('\t')
        if len(_cols) < 9 or _cols[2] != 'gene': continue
        _m = _re.search(r'ID=([^;]+)', _cols[8])
        if not _m: continue
        _gid = _m.group(1)
        # Prodigal predictions are all protein_coding; ncRNA genes have _ncRNA suffix
        if '_ncRNA' in _gid:
            gene_biotype[_gid] = 'ncRNA'
        else:
            gene_biotype[_gid] = 'protein_coding'
        _nm = _re.search(r'gene=([^;]+)', _cols[8])
        if not _nm:
            _nm = _re.search(r'Name=([^;]+)', _cols[8])
        if _nm:
            gene_name_map[_gid] = _nm.group(1)
print(f'✅ gene_biotype: {sum(1 for v in gene_biotype.values() if v=="protein_coding")} protein_coding, '
      f'{sum(1 for v in gene_biotype.values() if v!="protein_coding")} non-coding')
print(f'   gene_name_map: {len(gene_name_map)} / {len(gene_biotype)} named')

In [ ]:
# Load pco pre-UMAP h5ads (have X_umap + leiden + gene_name)
pco_md = sc.read_h5ad('pco/pco_md_umap.h5ad')
pco_hc = sc.read_h5ad('pco/pco_hc_umap.h5ad')
print(f'pco MD: {pco_md.shape}, genes with gene_name: {"gene_name" in pco_md.var.columns}')
print(f'pco HC: {pco_hc.shape}, genes with gene_name: {"gene_name" in pco_hc.var.columns}')
print(f'  MD obsm: {list(pco_md.obsm.keys())}')
print(f'  HC obsm: {list(pco_hc.obsm.keys())}')

In [ ]:
# Annotate calling status
pco_ref_md, pco_ref_hc, pco_shared, pco_md_only, pco_hc_only = annotate_calling_status(pco_md, pco_hc)
print('pco MD reference:')
print(pco_ref_md.obs['calling_status'].value_counts())
print()
print('pco HC reference:')
print(pco_ref_hc.obs['calling_status'].value_counts())

In [ ]:
# UMAP — calling status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_status_umap(pco_ref_md, pco_ref_hc, 'pco', axes)
plt.tight_layout()
plt.show()
#note4fig: fig5j

In [ ]:
# UMAP — Leiden
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sc.pl.umap(pco_ref_md, color='leiden', ax=axes[0], show=False, size=30, frameon=True,
           legend_loc='on data', title='pco MD — Leiden')
sc.pl.umap(pco_ref_hc, color='leiden', ax=axes[1], show=False, size=30, frameon=True,
           legend_loc='on data', title='pco HC — Leiden')
for ax in axes:
    for collection in ax.collections:
        collection.set_rasterized(True)
plt.tight_layout()
plt.show()
#note4fig: fig5i

In [ ]:
def gene_stats(adata):
    """Compute per-gene mean expression and fraction expressed."""
    X = adata.X
    if hasattr(X, 'toarray'):
        X = X.toarray()
    else:
        X = np.asarray(X)
    mean_expression = np.mean(X, axis=0)
    fraction_expressed = np.mean(X > 0, axis=0)
    result = pd.DataFrame({
        'gene_id': adata.var_names,
        'gene_name': adata.var['gene_name'].values,
        'mean_expression': mean_expression,
        'fraction_expressed': fraction_expressed
    })
    return result
pco_md_stats = gene_stats(pco_md)
pco_hc_stats = gene_stats(pco_hc)
print('=== pco MD top 20 expressed genes ===')
print(pco_md_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))
print()
print('=== pco HC top 20 expressed genes ===')
print(pco_hc_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))

In [ ]:
# Leiden marker genes (pco MD)
if pco_md.obs['leiden'].nunique() > 1:
    sc.tl.rank_genes_groups(pco_md, groupby='leiden', method='wilcoxon', key_added='leiden_markers')
    pco_md_markers = sc.get.rank_genes_groups_df(pco_md, group=None, key='leiden_markers')
    pco_md_markers['gene_name'] = pco_md_markers['names'].map(
        pd.Series(pco_md.var['gene_name'].values, index=pco_md.var_names)
    )
    print('pco MD — top 5 markers per cluster:')
    for cl in sorted(pco_md.obs['leiden'].unique(), key=lambda x: int(x)):
        top = pco_md_markers[pco_md_markers['group'] == cl].head(5)
        print(f'  Cluster {cl}: ' + ', '.join(top['gene_name'].tolist()))
else:
    print('pco MD: only 1 cluster, skipping marker gene analysis')

In [ ]:
# Leiden marker genes (pco HC)
if pco_hc.obs['leiden'].nunique() > 1:
    sc.tl.rank_genes_groups(pco_hc, groupby='leiden', method='wilcoxon', key_added='leiden_markers')
    pco_hc_markers = sc.get.rank_genes_groups_df(pco_hc, group=None, key='leiden_markers')
    pco_hc_markers['gene_name'] = pco_hc_markers['names'].map(
        pd.Series(pco_hc.var['gene_name'].values, index=pco_hc.var_names)
    )
    print('pco HC — top 5 markers per cluster:')
    for cl in sorted(pco_hc.obs['leiden'].unique(), key=lambda x: int(x)):
        top = pco_hc_markers[pco_hc_markers['group'] == cl].head(5)
        print(f'  Cluster {cl}: ' + ', '.join(top['gene_name'].tolist()))
else:
    print('pco HC: only 1 cluster, skipping marker gene analysis')

In [ ]:
top_genes = pco_md_stats.sort_values('mean_expression', ascending=False).head(10)['gene_name'].tolist()
print(f'Plotting {len(top_genes)} genes: ' + ', '.join(top_genes))
def gene_bubble_data(adata, genes, method):
    records = []
    for gene in genes:
        idx = np.where(adata.var['gene_name'].values == gene)[0]
        if len(idx) == 0:
            continue
        x = adata.X[:, idx[0]]
        if hasattr(x, 'toarray'):
            x = x.toarray().ravel()
        else:
            x = np.asarray(x).ravel()
        records.append(
            {'gene': gene, 'method': method, 'fraction_expressed': np.mean(x > 0), 'mean_expression': np.mean(x)}
        )
    return pd.DataFrame(records)
pco_bubble = pd.concat(
    [gene_bubble_data(pco_md, top_genes, 'ModuleDrops'), gene_bubble_data(pco_hc, top_genes, 'HardCutoff')],
    ignore_index=True,
)

vmin = 0
vmax = 1
methods = ['ModuleDrops', 'HardCutoff']
fig, ax = plt.subplots(figsize=(8, 2.8))
for yi, method in enumerate(methods):
    tmp = pco_bubble[pco_bubble['method'] == method]
    x = [top_genes.index(g) for g in tmp['gene']]
    scatter = ax.scatter(
        x,
        [yi] * len(tmp),
        s=tmp['fraction_expressed'] * 500,
        c=tmp['mean_expression'],
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
    )

ax.set_xticks(range(len(top_genes)))
ax.set_xticklabels(top_genes, rotation=45, ha='right')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xlabel('Gene')
ax.set_ylabel('Calling method')
ax.set_xlim(-0.5, len(top_genes) - 0.5)
ax.set_ylim(-0.5, len(methods) - 0.5)

cbar = plt.colorbar(scatter, ax=ax, pad=0.08, shrink=0.9, aspect=15)
cbar.set_label('Mean log1p normalized expression')

size_values = [0, 25, 50, 100]
size_handles = [
    ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
    for pct in size_values
]
ax.legend(
    handles=size_handles,
    title='Fraction expressed',
    loc='upper left',
    bbox_to_anchor=(1.02, 1.0),
    frameon=False,
    scatterpoints=1,
)

ax.set_title('pco — top expressed genes')
plt.tight_layout()
plt.show()

In [ ]:
# Gene bubble plot: Gene × Method (flipped)
# x = calling method
# y = gene
top_genes = pco_md_stats.sort_values('mean_expression', ascending=False).head(10)['gene_name'].tolist()
print(f'Plotting {len(top_genes)} genes: ' + ', '.join(top_genes))
def gene_bubble_data(adata, genes, method):
    records = []
    for gene in genes:
        idx = np.where(adata.var['gene_name'].values == gene)[0]
        if len(idx) == 0:
            continue
        x = adata.X[:, idx[0]]
        if hasattr(x, 'toarray'):
            x = x.toarray().ravel()
        else:
            x = np.asarray(x).ravel()
        records.append(
            {'gene': gene, 'method': method, 'fraction_expressed': np.mean(x > 0), 'mean_expression': np.mean(x)}
        )
    return pd.DataFrame(records)
pco_bubble = pd.concat(
    [gene_bubble_data(pco_md, top_genes, 'ModuleDrops'), gene_bubble_data(pco_hc, top_genes, 'HardCutoff')],
    ignore_index=True,
)

vmin = 0
vmax = 1
methods = ['ModuleDrops', 'HardCutoff']
fig, ax = plt.subplots(figsize=(5, 7))
for xi, method in enumerate(methods):
    tmp = pco_bubble[pco_bubble['method'] == method]
    y = [top_genes.index(g) for g in tmp['gene']]
    scatter = ax.scatter(
        [xi] * len(tmp),  # x = method
        y,  # y = gene
        s=tmp['fraction_expressed'] * 500,
        c=tmp['mean_expression'],
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
    )

ax.set_yticks(range(len(top_genes)))
ax.set_yticklabels(top_genes)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.set_xlabel('Calling method')
ax.set_ylabel('Gene')
ax.set_xlim(-0.5, len(methods) - 0.5)
ax.set_ylim(-0.5, len(top_genes) - 0.5)
ax.invert_yaxis()

cbar = plt.colorbar(scatter, ax=ax, pad=0.08, shrink=0.9, aspect=30)
cbar.set_label('Mean log1p normalized expression')

size_values = [0, 25, 50, 100]
size_handles = [
    ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
    for pct in size_values
]
ax.legend(
    handles=size_handles,
    title='Fraction expressed',
    loc='upper left',
    bbox_to_anchor=(1.02, 1.0),
    frameon=False,
    scatterpoints=1,
)

ax.set_title('pco — top expressed genes')
plt.tight_layout()
plt.show()
#note4fig: fig5h

In [ ]:

het_genes = []
if pco_md.obs['leiden'].nunique() > 1:
    for cl in sorted(pco_md.obs['leiden'].unique(), key=lambda x: int(x)):
        top = pco_md_markers[(pco_md_markers['group'] == cl) & (pco_md_markers['logfoldchanges'] > 0.5)]
        for g in top.head(2)['gene_name']:
            if g not in het_genes:
                het_genes.append(g)
    print(f'Initial heterogeneity genes ({len(het_genes)}): ' + ', '.join(het_genes))
    records = []
    gene_names = pco_md.var['gene_name'].values
    for cluster in sorted(pco_md.obs['leiden'].unique(), key=lambda x: int(x)):
        cells_mask = pco_md.obs['leiden'] == cluster
        adata_cluster = pco_md[cells_mask]
        for gene in het_genes:
            idx = np.where(gene_names == gene)[0]
            if len(idx) == 0:
                continue
            x = adata_cluster.X[:, idx[0]]
            if hasattr(x, 'toarray'):
                x = x.toarray().ravel()
            else:
                x = np.asarray(x).ravel()
            records.append(
                {'cluster': cluster, 'gene': gene, 'fraction_expressed': np.mean(x > 0), 'mean_expression': np.mean(x)}
            )
    bubble_het = pd.DataFrame(records)
    # 3. Filter genes: remove genes with <5% expression in ALL clusters
    MIN_FRACTION = 0.05
    gene_max_fraction = bubble_het.groupby('gene')['fraction_expressed'].max()
    het_genes = [g for g in het_genes if gene_max_fraction.get(g, 0) >= MIN_FRACTION]
    bubble_het = bubble_het[bubble_het['gene'].isin(het_genes)].copy()
    print(f'After filtering (<{MIN_FRACTION:.0%} in all clusters): {len(het_genes)} genes')
    print(', '.join(het_genes))
    vmin = 0
    vmax = 3.5
    clusters = sorted(bubble_het['cluster'].unique(), key=lambda x: int(x))
    fig_w = max(8, len(clusters) * 0.8 + 3)
    fig_h = max(7, len(het_genes) * 0.6 + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    for xi, cluster in enumerate(clusters):
        tmp = bubble_het[bubble_het['cluster'] == cluster]
        y = [het_genes.index(g) for g in tmp['gene']]
        scatter = ax.scatter(
            [xi] * len(tmp),
            y,
            s=tmp['fraction_expressed'] * 500,
            c=tmp['mean_expression'],
            cmap='RdBu_r',
            vmin=vmin,
            vmax=vmax,
            alpha=0.85,
            edgecolors='black',
        )
    ax.set_xticks(range(len(clusters)))
    ax.set_xticklabels(clusters)
    ax.set_yticks(range(len(het_genes)))
    ax.set_yticklabels(het_genes)
    ax.set_xlabel('Leiden cluster')
    ax.set_ylabel('Gene')
    ax.set_title('pco MD — cluster heterogeneity')
    cbar = plt.colorbar(scatter, ax=ax, pad=0.03, shrink=0.7)
    cbar.set_label('Mean log1p normalized expression')
    size_values = [0, 25, 50, 100]
    size_handles = [
        ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
        for pct in size_values
    ]
    ax.legend(
        handles=size_handles,
        title='Fraction expressed',
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        frameon=False,
        scatterpoints=1,
    )
    plt.tight_layout()
    plt.show()
else:
    print('pco MD: only 1 cluster, skipping heterogeneity plot')
#note4fig: fig5k_part2

In [ ]:

het_genes_hc = []
if pco_hc.obs['leiden'].nunique() > 1:
    for cl in sorted(pco_hc.obs['leiden'].unique(), key=lambda x: int(x)):
        top = pco_hc_markers[(pco_hc_markers['group'] == cl) & (pco_hc_markers['logfoldchanges'] > 0.5)]
        for g in top.head(2)['gene_name']:
            if g not in het_genes_hc:
                het_genes_hc.append(g)
    print(f'Initial HC heterogeneity genes ({len(het_genes_hc)}): ' + ', '.join(het_genes_hc))
    records_hc = []
    gene_names_hc = pco_hc.var['gene_name'].values
    for cluster in sorted(pco_hc.obs['leiden'].unique(), key=lambda x: int(x)):
        cells_mask = pco_hc.obs['leiden'] == cluster
        adata_cluster = pco_hc[cells_mask]
        for gene in het_genes_hc:
            idx = np.where(gene_names_hc == gene)[0]
            if len(idx) == 0:
                continue
            x = adata_cluster.X[:, idx[0]]
            if hasattr(x, 'toarray'):
                x = x.toarray().ravel()
            else:
                x = np.asarray(x).ravel()
            records_hc.append(
                {'cluster': cluster, 'gene': gene, 'fraction_expressed': np.mean(x > 0), 'mean_expression': np.mean(x)}
            )
    bubble_het_hc = pd.DataFrame(records_hc)
    # 3. Filter genes: remove genes whose fraction is <5% in ALL clusters
    MIN_FRACTION = 0.05
    gene_max_fraction_hc = bubble_het_hc.groupby('gene')['fraction_expressed'].max()
    het_genes_hc = [g for g in het_genes_hc if gene_max_fraction_hc.get(g, 0) >= MIN_FRACTION]
    bubble_het_hc = bubble_het_hc[bubble_het_hc['gene'].isin(het_genes_hc)].copy()
    print(f'\nAfter filtering (<{MIN_FRACTION:.0%} in all clusters): {len(het_genes_hc)} genes')
    print(', '.join(het_genes_hc))
    if len(het_genes_hc) == 0:
        raise ValueError('No genes passed the 5% fraction-expressed filter.')
    vmin_hc = 0
    vmax_hc = 3.5
    clusters_hc = sorted(bubble_het_hc['cluster'].unique(), key=lambda x: int(x))
    fig_w = max(8, len(clusters) * 0.8 + 3)
    fig_h = max(7, len(het_genes) * 0.6 + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    for xi, cluster in enumerate(clusters_hc):
        tmp = bubble_het_hc[bubble_het_hc['cluster'] == cluster]
        y = [het_genes_hc.index(g) for g in tmp['gene']]
        scatter = ax.scatter(
            [xi] * len(tmp),
            y,
            s=tmp['fraction_expressed'] * 500,
            c=tmp['mean_expression'],
            cmap='RdBu_r',
            vmin=vmin_hc,
            vmax=vmax_hc,
            alpha=0.85,
            edgecolors='black',
        )
    ax.set_xticks(range(len(clusters_hc)))
    ax.set_xticklabels(clusters_hc)
    ax.set_yticks(range(len(het_genes_hc)))
    ax.set_yticklabels(het_genes_hc)
    ax.set_xlabel('Leiden cluster')
    ax.set_ylabel('Gene')
    ax.set_title('pco HC — cluster heterogeneity')
    cbar = plt.colorbar(scatter, ax=ax, pad=0.03, shrink=0.7)
    cbar.set_label('Mean log1p normalized expression')
    size_values = [0, 25, 50, 100]
    size_handles = [
        ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
        for pct in size_values
    ]
    ax.legend(
        handles=size_handles,
        title='Fraction expressed',
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        frameon=False,
        scatterpoints=1,
    )
    plt.tight_layout()
    plt.show()
else:
    print('pco HC: only 1 cluster, skipping heterogeneity plot')
#note4fig: fig5k_part1

In [ ]:
# Purity comparison: method-only barcodes (from BAM)
# purity = mapped reads / total reads per barcode
import pandas as pd
import numpy as np
import seaborn as sns
# ---- Load BAM-based purity per barcode ----
pco_purity = pd.read_csv("pco/purity.tsv", sep="\t")
pco_pur_map = pco_purity.set_index("bc")["purity"]
print(f"pco barcodes with purity: {len(pco_pur_map)}")
# ---- Look up purity for method-only barcodes ----
md_only_purity = pco_pur_map.reindex(list(pco_md_only)).dropna().values
hc_only_purity = pco_pur_map.reindex(list(pco_hc_only)).dropna().values
print(
    f"\nMD-only: {len(pco_md_only)} barcodes, {len(md_only_purity)} with purity, median={np.median(md_only_purity):.4f}"
)
print(
    f"HC-only: {len(pco_hc_only)} barcodes, {len(hc_only_purity)} with purity, median={np.median(hc_only_purity):.4f}"
)
# ---- Violin plot ----
plot_data = pd.DataFrame(
    [{"method": "MD-only", "purity": v} for v in md_only_purity]
    + [{"method": "HC-only", "purity": v} for v in hc_only_purity]
)
fig, ax = plt.subplots(figsize=(4, 5), dpi=150)
colors = {"MD-only": "#4C78A8", "HC-only": "#E45756"}
sns.violinplot(data=plot_data, x="method", y="purity", palette=colors, inner="box", cut=0, ax=ax)
ax.set(ylabel="Purity (mapped reads / total reads)", xlabel="Calling method", title="Purity of method-only barcodes")
ax.set_ylim(0, 1.05)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig5f

In [ ]:
# Compare MD vs HC marker genes for Prevotella copri_A
# Volcano-style scatter: x=log2FC, y=-log10(padj)
# Orange = MD-only, Blue = shared MD, Red = shared HC
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
MD_TSV = "ko_species_markers.md.tsv"
HC_TSV = "ko_species_markers.hc.tsv"
TARGET_SPECIES = "Prevotella copri_A"

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 5
plt.rcParams["axes.labelsize"] = 5
plt.rcParams["axes.titlesize"] = 5
plt.rcParams["xtick.labelsize"] = 5
plt.rcParams["ytick.labelsize"] = 5
plt.rcParams["legend.fontsize"] = 5
plt.rcParams["figure.titlesize"] = 5
md_df = pd.read_csv(MD_TSV, sep="\t")
hc_df = pd.read_csv(HC_TSV, sep="\t")
# Filter to target species
md_sp = md_df[md_df["species"] == TARGET_SPECIES].copy()
hc_sp = hc_df[hc_df["species"] == TARGET_SPECIES].copy()
print(f"MD markers for {TARGET_SPECIES}: {len(md_sp)}")
print(f"HC markers for {TARGET_SPECIES}: {len(hc_sp)}")
# Shared and unique genes
md_genes = set(md_sp["gene_name"].tolist())
hc_genes = set(hc_sp["gene_name"].tolist())
shared = md_genes & hc_genes
md_only = md_genes - hc_genes
hc_only = hc_genes - md_genes
print(f"Shared: {len(shared)}, MD-only: {len(md_only)}, HC-only: {len(hc_only)}")
# Compute -log10(padj)
md_sp["neg_log_padj"] = -np.log10(md_sp["padj"].clip(lower=1e-300))
hc_sp["neg_log_padj"] = -np.log10(hc_sp["padj"].clip(lower=1e-300))
# Index by gene_name for lookup
md_idx = md_sp.set_index("gene_name")
hc_idx = hc_sp.set_index("gene_name")
# Plot
fig, ax = plt.subplots(figsize=(2, 1.4), dpi=300)

for spine in ax.spines.values():
    spine.set_linewidth(0.3)
ax.tick_params(axis="both", which="major", direction="out",
               width=0.3, length=2, pad=2)
ax.tick_params(axis="both", which="minor", direction="out",
               width=0.2, length=1)
# MD-only (orange)
md_only_df = md_sp[md_sp["gene_name"].isin(md_only)]
ax.scatter(
    md_only_df["fc"], md_only_df["neg_log_padj"],
    c="orange", s=10, alpha=1,
    edgecolor="none", linewidths=0,
    label=f"MD only ({len(md_only)})",
    rasterized=True, zorder=3
)
# Shared genes: draw connecting lines + larger points
shared_md = md_sp[md_sp["gene_name"].isin(shared)].sort_values("gene_name")
shared_hc = hc_sp[hc_sp["gene_name"].isin(shared)].sort_values("gene_name")
for _, row in shared_md.iterrows():
    gn = row["gene_name"]
    md_x, md_y = row["fc"], row["neg_log_padj"]
    hc_row = hc_idx.loc[gn]
    hc_x, hc_y = hc_row["fc"], hc_row["neg_log_padj"]
    # Connecting line
    ax.plot([md_x, hc_x], [md_y, hc_y], color="black", linewidth=0.5,
            alpha=0.6, zorder=4, linestyle="--")
# Shared MD points (blue)
ax.scatter(
    shared_md["fc"], shared_md["neg_log_padj"],
    c="#2166ac", s=10, alpha=0.9,
    edgecolor="none", linewidth=0,
    label=f"Shared ({len(shared)})", zorder=5, rasterized=True
)
# Shared HC points (red)
ax.scatter(
    shared_hc["fc"], shared_hc["neg_log_padj"],
    c="#b2182b", s=10, alpha=0.9,
    edgecolor="none", linewidth=0,
    zorder=6, rasterized=True,
    label='Shared HC'
)
ax.set_xlabel("Log2FC")
ax.set_ylabel("-log10padj")
ax.set_title(f"{TARGET_SPECIES}: MD vs HC marker genes")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
#note4fig: fig5g